# Food101-LT imbalance sweep

Measures how the benefit of a **fixed** synthetic budget (3,800 images) varies
with the imbalance ratio of the training set.

Five ratios are built from the same Food-101 source with an identical class
ordering, so the class that is largest at IR=150 is also largest at IR=1 --
only the steepness of the decay changes. The 3,800 synthetic images are the
same files at every ratio.

| IR | real | synth | frac |
|---|---|---|---|
| 150 | 15,244 | 3,800 | 24.9% |
| 50 | 19,172 | 3,800 | 19.8% |
| 20 | 24,268 | 3,800 | 15.7% |
| 5 | 37,728 | 3,800 | 10.1% |
| 1 | 75,750 | 3,800 | 5.0% |

**Known confound:** holding the budget fixed means the synthetic *fraction*
falls as IR falls, so the curve mixes "less imbalance" with "more dilution".
Both push the same way, so a decaying gain bounds the imbalance effect from
above. The UEC-256 result (IR 7.3 at 24% synthetic, gain ~0) is the
independent check that argues imbalance rather than dilution is doing the work.

IR=150 arms A and B already exist under the tags `A_align_s*` / `B_align_s*`
and are **not** rerun here.

In [ ]:
# coding: utf-8
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import copy, csv, json, math, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import swin_b

FOOD101    = Path("/workspace/food101/food-101")
LT_DIR     = Path("/workspace/food101_lt")
ARMS       = Path("/workspace/food101_lt_arms")
POOL       = Path("/workspace/food101_lt_synth")
ALIGN_CKPT = "/workspace/align/keep/align_train_20k_seed42.pth"

DEVICE = "cuda"
SEED   = 42
EPOCHS, BATCH, LR = 30, 32, 1e-4
WARMUP, WD, MIXUP, SMOOTH, EMA_DECAY, TTA_VIEWS, WORKERS = 3, 1e-4, 0.2, 0.1, 0.999, 8, 12
BUDGET = 3800

SWEEP  = Path("/workspace/food101_lt/sweep")
FIXED_BUDGET = 3800
_SWEEP_SPLIT = "train_lt"     # patched by run_sweep(); default = IR 150

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

free, total = torch.cuda.mem_get_info()
print(f"torch {torch.__version__}   {free/1e9:.1f} GB free of {total/1e9:.1f}")
SWEEP_M = json.loads((SWEEP/"manifest.json").read_text())
print(f"sweep ratios: "
      + "  ".join(f'IR{m["imbalance_ratio"]}={m["total_train_images"]}'
                  for m in SWEEP_M["ratios"]))

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3,0.05),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
    transforms.RandomErasing(p=0.25)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
tta_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

class Food101LT(Dataset):
    def __init__(self, split, tf, synth_items=None):
        self.classes = sorted((FOOD101/"meta"/"classes.txt").read_text().split())
        self.c2i = {c: i for i, c in enumerate(self.classes)}
        self.tf = tf
        self.items = [(FOOD101/"images"/f"{l}.jpg", self.c2i[l.split("/")[0]])
                      for l in (FOOD101/"meta"/f"{split}.txt").read_text().split()]
        self.n_real = len(self.items)
        self.n_synth = 0
        if synth_items:
            self.items.extend(synth_items); self.n_synth = len(synth_items)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def load_synth_list(path, c2i):
    if not path: return None
    items = []
    for ln in Path(path).read_text().splitlines():
        if not ln.strip(): continue
        img, cls = ln.split("\t")
        items.append((Path(img), c2i[cls]))
    return items

seed_all()
CLASSES = sorted((FOOD101/"meta"/"classes.txt").read_text().split())
C2I = {c: i for i, c in enumerate(CLASSES)}
def set_head_tail(ir):
    """head/tail membership changes with the imbalance ratio, so it is
    reset before every run rather than fixed once."""
    global HEAD_IDS, TAIL_IDS
    f = (LT_DIR/"class_counts.csv") if ir == 150 else (SWEEP/f"class_counts_ir{ir}.csv")
    g = {r["class"]: r["group"] for r in csv.DictReader(open(f))}
    HEAD_IDS = torch.tensor([i for i, c in enumerate(CLASSES) if g[c] == "head"])
    TAIL_IDS = torch.tensor([i for i, c in enumerate(CLASSES) if g[c] == "tail"])
    return len(HEAD_IDS), len(TAIL_IDS)

set_head_tail(150)

te, te_tta = Food101LT("test", eval_tf), Food101LT("test", tta_tf)
mk = lambda d, s: DataLoader(d, batch_size=BATCH, shuffle=s,
                             num_workers=WORKERS, pin_memory=True)
te_dl, tta_dl = mk(te, False), mk(te_tta, False)
print(f"test {len(te)}   classes {len(CLASSES)}   head {len(HEAD_IDS)}   tail {len(TAIL_IDS)}")

In [ ]:
def build_swin(n_classes, init):
    m = swin_b(weights="IMAGENET1K_V1" if init == "imagenet" else None)
    dim = m.head.in_features
    m.head = nn.Identity()
    if init == "align":
        sd = torch.load(ALIGN_CKPT, map_location="cpu")
        sd = sd.get("state_dict", sd.get("model", sd))
        ext = {k[len("visual."):]: v for k, v in sd.items() if k.startswith("visual.")}
        missing, _ = m.load_state_dict(ext, strict=False)
        loaded = len(m.state_dict()) - len(missing)
        print(f"alignment init: {loaded}/{len(m.state_dict())} tensors")
        assert loaded > 0.8*len(m.state_dict()), "prefix mismatch"
    m.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim, n_classes))
    return m

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.mul_(self.decay).add_(m.detach(), alpha=1-self.decay)
        for s, m in zip(self.shadow.buffers(), model.buffers()):
            s.copy_(m)

def mixup(x, y, alpha=MIXUP):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

In [ ]:
def split_acc(c1, c5, labels):
    head_m = torch.isin(labels, HEAD_IDS); tail_m = ~head_m
    def pct(mask, c):
        n = int(mask.sum().item())
        return round(100.0 * c[mask].sum().item() / max(n, 1), 2)
    all_m = torch.ones_like(head_m)
    return {"top1": pct(all_m, c1), "top5": pct(all_m, c5),
            "top1_head": pct(head_m, c1), "top1_tail": pct(tail_m, c1),
            "top5_head": pct(head_m, c5), "top5_tail": pct(tail_m, c5)}

@torch.no_grad()
def evaluate(model, loader, want_cm=False):
    model.eval(); c1, c5, ls, pr = [], [], [], []
    for x, y in loader:
        top5 = model(x.to(DEVICE, non_blocking=True)).topk(5, 1, True, True)[1].cpu()
        c = top5.eq(y.view(-1,1))
        c1.append(c[:,0]); c5.append(c.any(1)); ls.append(y); pr.append(top5[:,0])
    labels = torch.cat(ls)
    out = split_acc(torch.cat(c1), torch.cat(c5), labels)
    if not want_cm: return out, None
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    np.add.at(cm, (labels.numpy(), torch.cat(pr).numpy()), 1)
    return out, cm

@torch.no_grad()
def evaluate_tta(model, base, tta, n_views=TTA_VIEWS, want_cm=False):
    model.eval(); probs = labels = None
    for v in range(n_views):
        loader = base if v == 0 else tta
        ps, ls = [], []
        for x, y in loader:
            ps.append(F.softmax(model(x.to(DEVICE, non_blocking=True)), 1).cpu()); ls.append(y)
        vp = torch.cat(ps)
        probs = vp if probs is None else probs + vp
        if labels is None: labels = torch.cat(ls)
    top5 = probs.topk(5, 1, True, True)[1]
    c = top5.eq(labels.view(-1,1))
    out = split_acc(c[:,0], c.any(1), labels)
    if not want_cm: return out, None
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    np.add.at(cm, (labels.numpy(), top5[:,0].numpy()), 1)
    return out, cm

In [ ]:
def run_arm(tag, init="align", synth=None, alloc="none",
            expect_synth=None, seed=SEED):
    seed_all(seed)
    tr = Food101LT(_SWEEP_SPLIT, train_tf, load_synth_list(synth, C2I))
    if expect_synth is not None and tr.n_synth != expect_synth:
        raise SystemExit(f"budget mismatch: {tr.n_synth} synthetic, expected {expect_synth}")
    tr_dl = mk(tr, True)
    print(f"{tag}:  split {_SWEEP_SPLIT}   real {tr.n_real}   "
      f"synth {tr.n_synth}   total {len(tr)}   seed {seed}")

    model = build_swin(len(CLASSES), init).to(DEVICE)
    ema   = EMA(model)
    crit  = nn.CrossEntropyLoss(label_smoothing=SMOOTH)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler("cuda")

    steps = len(tr_dl)
    def lr_at(ep, it):
        t = ep + it/steps
        if t < WARMUP: return LR * t / WARMUP
        return LR * 0.5 * (1 + math.cos(math.pi * (t-WARMUP)/max(EPOCHS-WARMUP, 1)))

    history, best, t0 = [], -1.0, time.time()
    for ep in range(EPOCHS):
        model.train(); run, seen = 0.0, 0
        for it, (x, y) in enumerate(tr_dl):
            for g in opt.param_groups: g["lr"] = lr_at(ep, it)
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            xm, ya, yb, lam = mixup(x, y)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                o = model(xm)
                loss = lam*crit(o, ya) + (1-lam)*crit(o, yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            ema.update(model)
            run += loss.item()*y.size(0); seen += y.size(0)

        m, _ = evaluate(ema.shadow, te_dl)
        history.append({"epoch": ep+1, "loss": round(run/seen, 4), **m})
        print(f"  ep {ep+1:>2}/{EPOCHS}  loss {run/seen:.4f}  top1 {m['top1']:6.2f}  "
              f"head {m['top1_head']:6.2f}  tail {m['top1_tail']:6.2f}  ({time.time()-t0:.0f}s)")
        if m["top1"] > best:
            best = m["top1"]; torch.save(ema.shadow.state_dict(), ARMS/f"{tag}.pth")

    model.load_state_dict(torch.load(ARMS/f"{tag}.pth", map_location=DEVICE)); model.to(DEVICE)
    no_tta,  cm  = evaluate(model, te_dl, want_cm=True)
    with_tta, cmt = evaluate_tta(model, te_dl, tta_dl, want_cm=True)
    np.save(ARMS/f"{tag}_confusion.npy", cm)
    np.save(ARMS/f"{tag}_confusion_tta.npy", cmt)

    res = {"tag": tag, "init": init, "alloc": alloc, "synth_list": str(synth),
           "n_real": tr.n_real, "n_synth": tr.n_synth,
           "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "warmup": WARMUP,
                      "wd": WD, "mixup": MIXUP, "smooth": SMOOTH, "ema": EMA_DECAY,
                      "tta_views": TTA_VIEWS, "seed": seed, "budget": FIXED_BUDGET,
                      "split": _SWEEP_SPLIT},
           "no_tta": no_tta, "with_tta": with_tta,
           "minutes": round((time.time()-t0)/60, 1), "history": history}
    (ARMS/f"{tag}_results.json").write_text(json.dumps(res, indent=2))

    print(f"\n{tag}")
    for name, m in (("no TTA", no_tta), ("8-view TTA", with_tta)):
        print(f"  {name:<12} top1 {m['top1']:6.2f}   head {m['top1_head']:6.2f}   "
              f"tail {m['top1_tail']:6.2f}   top5 {m['top5']:6.2f}")
    print(f"  {res['minutes']} min\n")
    return res

## Synthetic list — 3,800 images, uniform across classes, shared by every ratio

In [ ]:
pool_have = np.array([len(list((POOL/c).glob("*.jpg"))) for c in CLASSES])
print(f"pool holds {pool_have.sum()} images")

def uniform_alloc(budget, have):
    a = np.full(len(have), budget // len(have), dtype=int)
    for i in range(budget - int(a.sum())): a[i] += 1
    a = np.minimum(a, have)
    deficit = budget - int(a.sum())
    for ci in np.argsort(-(have - a)):
        if deficit <= 0: break
        take = int(min(have[ci] - a[ci], deficit)); a[ci] += take; deficit -= take
    assert deficit == 0, f"pool short by {deficit}"
    return a

alloc = uniform_alloc(FIXED_BUDGET, pool_have)
lines = []
for ci, c in enumerate(CLASSES):
    for f in sorted((POOL/c).glob("*.jpg"))[:alloc[ci]]:
        lines.append(f"{f}\t{c}")

SYNTH_SWEEP = SWEEP/"synth_fixed3800.txt"
SYNTH_SWEEP.write_text("\n".join(lines) + "\n")
assert len(lines) == FIXED_BUDGET, len(lines)
print(f"{len(lines)} synthetic images -> {SYNTH_SWEEP}")
print(f"per class: min {alloc.min()}  max {alloc.max()}")

## Sweep runner

In [ ]:
def run_sweep(ir, arm, seed):
    """arm 'A' = real only, arm 'B' = real + 3800 uniform synthetic."""
    global _SWEEP_SPLIT
    nh, nt = set_head_tail(ir)
    _SWEEP_SPLIT = "train_lt" if ir == 150 else f"train_lt_ir{ir}"
    print(f"--- IR {ir}  arm {arm}  seed {seed}   head {nh} / tail {nt} ---")
    try:
        return run_arm(f"ir{ir}_{arm}_s{seed}", init="align",
                       synth=(SYNTH_SWEEP if arm == "B" else None),
                       alloc=("none" if arm == "A" else "uniform"),
                       expect_synth=(FIXED_BUDGET if arm == "B" else None),
                       seed=seed)
    finally:
        _SWEEP_SPLIT = "train_lt"
        set_head_tail(150)

## Sanity check — run this first

One run at IR=150 arm A seed 42. It must reproduce **71.92** top-1. If it
does, this notebook is equivalent to the one that produced the anchor point
and the sweep numbers are comparable. If it does not, stop and find out why
before spending 30 hours.

In [ ]:
_ = run_sweep(150, "A", 42)   # expect top1 71.92, real 15244

## Launch

16 runs. IR=1 has 75,750 training images so those four runs take roughly four
times longer than the rest — budget ~30 hours in total, not 21.

In [ ]:
TODO = [(ir, arm, s) for ir in (50, 20, 5, 1)
                     for arm in ("A", "B")
                     for s in (42, 1)]
for ir, arm, s in TODO:
    print(f"  IR {ir:<4} {arm}  seed {s}")
print(f"\n{len(TODO)} runs")

In [ ]:
for ir, arm, s in TODO:
    run_sweep(ir, arm, s)

## Collect

In [ ]:
import glob
res = {}
for f in sorted(glob.glob(str(ARMS/"ir*_results.json"))):
    r = json.load(open(f))
    ir, arm, _ = r["tag"].split("_")
    res.setdefault((int(ir[2:]), arm), []).append(r["no_tta"])
for f in sorted(glob.glob(str(ARMS/"[AB]_align_s*_results.json"))):
    r = json.load(open(f))
    res.setdefault((150, r["tag"][0]), []).append(r["no_tta"])

print(f"{'IR':>4} {'frac':>6}  {'A real only':>14}  {'B +3800 synth':>14}  {'gain':>7}  {'tail gain':>10}")
for ir in sorted({k[0] for k in res}, reverse=True):
    A = res.get((ir, "A"), []); B = res.get((ir, "B"), [])
    if not A or not B: continue
    a1 = np.array([x["top1"] for x in A]); b1 = np.array([x["top1"] for x in B])
    at = np.array([x["top1_tail"] for x in A]); bt = np.array([x["top1_tail"] for x in B])
    real = {m["imbalance_ratio"]: m["total_train_images"] for m in SWEEP_M["ratios"]}[ir]
    print(f"{ir:>4} {FIXED_BUDGET/real:>5.1%}  {a1.mean():6.2f} ± {a1.std():.2f}  "
          f"{b1.mean():6.2f} ± {b1.std():.2f}  {b1.mean()-a1.mean():+7.2f}  "
          f"{bt.mean()-at.mean():+10.2f}")

In [ ]:
import glob, json
for f in sorted(glob.glob("/workspace/food101_lt_arms/ir*_results.json")):
    r = json.load(open(f))
    print(f"{r['tag']:<16} real {r['n_real']:>6}  synth {r['n_synth']:>5}  "
          f"top1 {r['no_tta']['top1']:6.2f}  tail {r['no_tta']['top1_tail']:6.2f}")